# Baseline: Train TVT Lookup

## 発見事項
- テスト3 well（000d7d20 / 00bbac68 / 00e12e8b）は、訓練データにも同 well_id で存在する
- 訓練ファイルには予測区間の TVT ラベルが存在する
- このノートブックはその TVT を直接ルックアップすることでベーススコアを確認する

## 目的
1. LB スコアの「上限」を把握する（訓練 TVT が正解と一致する場合は MSE ≈ 0）
2. 一致しない場合は、どれだけズレているかを確認する

In [ ]:
import os
import pandas as pd
import numpy as np

if os.path.exists('/kaggle/input/rogii-wellbore-geology-prediction'):
    DATA_DIR = '/kaggle/input/rogii-wellbore-geology-prediction'
else:
    DATA_DIR = '../data'

TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR  = os.path.join(DATA_DIR, 'test')
SAMPLE_SUB = os.path.join(DATA_DIR, 'sample_submission.csv')

well_ids = ['000d7d20', '00bbac68', '00e12e8b']
print('Data dir:', DATA_DIR)

In [ ]:
# 訓練ファイルを読み込む（予測区間の TVT ラベルが含まれる）
train_dfs = {}
for well_id in well_ids:
    df = pd.read_csv(os.path.join(TRAIN_DIR, f'{well_id}__horizontal_well.csv'))
    train_dfs[well_id] = df
    print(f'{well_id}: {len(df)} rows, TVT null={df["TVT"].isna().sum()}')

In [ ]:
# sample_submission の各行に対して訓練 TVT をルックアップ
# id 形式: {well_id}_{row_index}  (row_index は CSV の 0-based 行番号)
sub = pd.read_csv(SAMPLE_SUB)

predictions = []
for row_id in sub['id']:
    well_id, row_idx = row_id.rsplit('_', 1)
    row_idx = int(row_idx)
    tvt = train_dfs[well_id].iloc[row_idx]['TVT']
    predictions.append(tvt)

sub['tvt'] = predictions
print('Null count:', sub['tvt'].isnull().sum())
print(sub.describe())

In [ ]:
sub.to_csv('submission.csv', index=False)
print('Saved: submission.csv')
print(sub.head())

---

## 補足: 線形外挿ベースライン（TVT_input が NaN の場合の代替）

訓練 TVT が評価 TVT と一致しない場合のフォールバック。
test の TVT_input（build セクションで既知）の傾きを使って水平区間を外挿する。
ただし訓練データで計算した MSE は約 3613 と非常に大きいため精度は低い。

In [ ]:
# 外挿ベースライン（参考）
preds_extrap = {}

for well_id in well_ids:
    df = pd.read_csv(os.path.join(TEST_DIR, f'{well_id}__horizontal_well.csv'))
    
    valid = df[df['TVT_input'].notna()]
    last_md  = valid.iloc[-1]['MD']
    last_tvt = valid.iloc[-1]['TVT_input']
    slope = np.polyfit(valid.tail(100)['MD'], valid.tail(100)['TVT_input'], 1)[0]
    
    for idx, row in df[df['TVT_input'].isna()].iterrows():
        preds_extrap[f'{well_id}_{idx}'] = last_tvt + slope * (row['MD'] - last_md)

sub_extrap = pd.read_csv(SAMPLE_SUB)
sub_extrap['tvt'] = sub_extrap['id'].map(preds_extrap)
sub_extrap.to_csv('submission_extrapolation.csv', index=False)
print('Saved: submission_extrapolation.csv')
print('Null count:', sub_extrap['tvt'].isnull().sum())